Here is the table of lattice parameters used in our paper

In [1]:
import numpy as np
#a,c=3.9652, 6.7455#here is x=0.0
#a,c=3.9513100000000003, 6.6994125#here is x=0.15
a,c=3.9374199999999995, 6.653325 #here is x=0.3
#a,c=3.92353, 6.6072375
#a,c=3.9096399999999996, 6.56115
#a,c=3.89575, 6.5150625
#a,c=3.8818599999999996, 6.468975
#a,c=3.8679699999999997, 6.4228875
#a,c=3.8459774999999996, 6.3499156249999995
#a,c=3.8239849999999995, 6.27694375
#a,c=3.8019925, 6.203971875
#a,c=3.78, 6.131
#unfold comparison
#a,c=3.953625, 6.70709375
#a,c=3.94205, 6.6686875
#a,c=3.906,6.635#dft qe supercell
lattice = a*np.array([
               [ 1.000000  , 0.000000  , 0.000000], 
               [-0.500000  , np.sqrt(3)/2  , 0.000000], 
               [ 0.000000  , 0.000000  , c/a], 
])
z=0.25
#ZrM2
positions_atoms_m=[[0.3333333333    ,    0.6666666666   ,     z],[0.6666666666    ,    0.3333333333   ,    1-z ]]
positions_atoms_zr = [[0.0000000000    ,   -0.0000000000   ,     0.0000000000]]

positions = np.concatenate((np.array(positions_atoms_zr),np.array(positions_atoms_m)))
alpha=0.3
N=f"{alpha}"

In [2]:
#lets start runing the GPAW scf and nscf for ZrTe2 and ZrSe2:
from ase import Atoms
from gpaw import GPAW, PW, MixerSum
from irrep.spacegroup import SpaceGroup
seed_list = ["zrse2","zrte2"]
symbols_list = [["Zr", "Se", "Se"],["Zr", "Te", "Te"]]

for i in range(len(seed_list)):
    atoms = Atoms(
       symbols_list[i],
       cell=lattice,
       pbc=True,
       scaled_positions=positions

    )

    calc = GPAW(
       mode=PW(500),
       xc="PBE",
       kpts={"size": [6, 6, 4], "gamma": True},
       convergence={"density": 1e-6},
       mixer=MixerSum(0.25, 8, 100),
       txt=f"{seed_list[i]}-scf.txt"
    )

    atoms.calc = calc
    atoms.get_potential_energy()
    calc.write(f"{seed_list[i]}-scf.gpw", mode="all")


    from irrep.spacegroup import SpaceGroup
    sg = SpaceGroup.from_gpaw(calc)
    kpoints_irred = sg.get_irreducible_kpoints_grid([6, 6, 4])
    calc_nscf = calc.fixed_density(
        kpts=kpoints_irred,
        symmetry={'symmorphic': False},
        nbands=60,
        convergence={'bands': 40},
        txt=f'{seed_list[i]}-nscf.txt')
    calc_nscf.write(f'{seed_list[i]}-nscf{N}-irred-664.gpw', mode='all')


typat used for spacegroup detection (accounting magmoms): [40 34 34]
typat used for spacegroup detection (accounting magmoms): [40 52 52]


 Now lets wannierize BOTH compounds and obtain their real-space Hamiltonians!

In [4]:

from wannierberri.w90files.wandata_soc import WannierDataSOC
from wannierberri.symmetry.projections import Projection, ProjectionsSet
from wannierberri.system.system_soc import SystemSOC
system_list=[]
for i in range(2):
    calc_nscf_irred = GPAW(f'{seed_list[i]}-nscf{N}-irred-664.gpw', txt=None)
    Ef = calc_nscf_irred.get_fermi_level()
    Ef_te=Ef
    sg = SpaceGroup.from_gpaw(calc_nscf_irred)
    z=0.25


    proj_p = Projection(position_num=positions_atoms_m, orbital="p", spacegroup=sg)


    proj_d = Projection(position_num=positions_atoms_zr, orbital="d", spacegroup=sg)

    proj_set = ProjectionsSet([proj_p, proj_d])



    wandata = WannierDataSOC.from_gpaw(
        calculator=calc_nscf_irred,
        projections=proj_set,
        spacegroup=sg,irreducible=True
    )


    frozmin,frozmax=Ef-6,Ef+2.6


    wandata.wannierise(
        froz_min=frozmin,
        froz_max=frozmax,
        outer_min=-100,
        outer_max=50,
        num_iter=100,
        print_progress_every=10,
        sitesym=True,
        localise=True,
    )
    #The fullham flag is used to obtain the H_scalar +H_SOC = H_full, necessary for a suitable description of the system.
    system = SystemSOC.from_wannierdata(wandata=wandata, berry=True, silent=False,full_ham=True)
    system_list.append(system)

typat used for spacegroup detection (accounting magmoms): [40 34 34]
Using 'projections' for both spin up channel.
finding num points from 2 projections
finding num points from 2 projections
got irreducible=True, mp_grid=None, seedname=wannier_soc-spin-0, files=['mmn', 'eig', 'amn', 'symmetrizer'],  projections=ProjectionsSet with 11 Wannier functions and 0 free variables
Projection 0.3333333333, 0.6666666666, 0.25:['p'] with 6 Wannier functions (6 per spin x1 spins) 
 on 2 points (3 per site)
Projection 0.0, -0.0, 0.0:['d'] with 5 Wannier functions (5 per spin x1 spins) 
 on 1 points (5 per site), unk_grid=None, normalize=True
self.irreducible=True
mpgrid = [6 6 4], 24
Shells found with weights [8.97031568 3.53432318] and tolerance 1.5777428138773402e-16
Creating amn. Using projections_set 
ProjectionsSet with 11 Wannier functions and 0 free variables
Projection 0.3333333333, 0.6666666666, 0.25:['p'] with 6 Wannier functions (6 per spin x1 spins) 
 on 2 points (3 per site)
Projection 

/home/caua/miniconda3/envs/wb_env3/lib/python3.11/site-packages/wannierberri/symmetry/sawf_kirr.py:163: UserWarning: Warning: max error in block 37 [54:56] is 5.90945565238381e-06, exceeding threshold 1e-06, and this is not among the  upper bands(56:60) bands, this may indicate inaccuracy in the input data
  warnings.warn(f"Warning: max error in block {i} [{start}:{end}] is {max_error_in_blocks[i]}, exceeding threshold {accuracy_threshold}, and this is not among the  upper"
/home/caua/miniconda3/envs/wb_env3/lib/python3.11/site-packages/wannierberri/symmetry/sawf_kirr.py:163: UserWarning: Warning: max error in block 36 [51:52] is 9.790866808668482e-06, exceeding threshold 1e-06, and this is not among the  upper bands(56:60) bands, this may indicate inaccuracy in the input data
  warnings.warn(f"Warning: max error in block {i} [{start}:{end}] is {max_error_in_blocks[i]}, exceeding threshold {accuracy_threshold}, and this is not among the  upper"
/home/caua/miniconda3/envs/wb_env3/lib/py

Symmetrizer_Uirr initialized for ikirr=6, kpt=6, [0.         0.16666667 0.75      ] with 4 symmetries, max error in included blocks: 3.539369616185151e-06 ; excluded bands are [56 57 58 59] out of 60 total bands (accuracy threshold 1e-06)
Symmetrizer_Uirr initialized for ikirr=7, kpt=7, [0.         0.33333333 0.        ] with 4 symmetries, max error in included blocks: 1.8590578537013992e-06 ; excluded bands are [56 57 58 59] out of 60 total bands (accuracy threshold 1e-06)
Symmetrizer_Uirr initialized for ikirr=8, kpt=8, [0.         0.33333333 0.25      ] with 4 symmetries, max error in included blocks: 2.7238396995596095e-05 ; excluded bands are [56 57 58 59] out of 60 total bands (accuracy threshold 1e-06)
Symmetrizer_Uirr initialized for ikirr=9, kpt=9, [0.         0.33333333 0.5       ] with 4 symmetries, max error in included blocks: 6.395449413591433e-05 ; excluded bands are [56 57 58 59] out of 60 total bands (accuracy threshold 1e-06)
Symmetrizer_Uirr initialized for ikirr=10,

/home/caua/miniconda3/envs/wb_env3/lib/python3.11/site-packages/wannierberri/symmetry/sawf_kirr.py:163: UserWarning: Warning: max error in block 53 [53:54] is 1.2452706161859092e-06, exceeding threshold 1e-06, and this is not among the  upper bands(56:60) bands, this may indicate inaccuracy in the input data
  warnings.warn(f"Warning: max error in block {i} [{start}:{end}] is {max_error_in_blocks[i]}, exceeding threshold {accuracy_threshold}, and this is not among the  upper"
/home/caua/miniconda3/envs/wb_env3/lib/python3.11/site-packages/wannierberri/symmetry/sawf_kirr.py:163: UserWarning: Warning: max error in block 54 [54:55] is 5.763446014208587e-06, exceeding threshold 1e-06, and this is not among the  upper bands(56:60) bands, this may indicate inaccuracy in the input data
  warnings.warn(f"Warning: max error in block {i} [{start}:{end}] is {max_error_in_blocks[i]}, exceeding threshold {accuracy_threshold}, and this is not among the  upper"
/home/caua/miniconda3/envs/wb_env3/lib/

Symmetrizer_Uirr initialized for ikirr=16, kpt=16, [0.16666667 0.16666667 0.5       ] with 4 symmetries, max error in included blocks: 3.4632854733280915e-06 ; excluded bands are [56 57 58 59] out of 60 total bands (accuracy threshold 1e-06)
Symmetrizer_Uirr initialized for ikirr=21, kpt=21, [0.33333333 0.33333333 0.        ] with 12 symmetries, max error in included blocks: 1.4639389985366986e-06 ; excluded bands are [57 58 59] out of 60 total bands (accuracy threshold 1e-06)
Symmetrizer_Uirr initialized for ikirr=22, kpt=22, [0.33333333 0.33333333 0.25      ] with 6 symmetries, max error in included blocks: 1.6142882225842236e-06 ; excluded bands are [55 56 57 58 59] out of 60 total bands (accuracy threshold 1e-06)
####################################################################################################
starting WFs
----------------------------------------------------------------------------------------------------
wannier centers and spreads
------------------------------

...

Symmetrizer_Uirr initialized for ikirr=6, kpt=6, [0.         0.16666667 0.75      ] with 4 symmetries, max error in included blocks: 3.755607543730463e-05 ; excluded bands are [56 57 58 59] out of 60 total bands (accuracy threshold 1e-06)
Symmetrizer_Uirr initialized for ikirr=7, kpt=7, [0.         0.33333333 0.        ] with 4 symmetries, max error in included blocks: 6.915363915601889e-06 ; excluded bands are [57 58 59] out of 60 total bands (accuracy threshold 1e-06)
Symmetrizer_Uirr initialized for ikirr=8, kpt=8, [0.         0.33333333 0.25      ] with 4 symmetries, max error in included blocks: 0.004691273826404718 ; excluded bands are [56 57 58 59] out of 60 total bands (accuracy threshold 1e-06)
Symmetrizer_Uirr initialized for ikirr=9, kpt=9, [0.         0.33333333 0.5       ] with 4 symmetries, max error in included blocks: 3.115436248836774e-05 ; excluded bands are [56 57 58 59] out of 60 total bands (accuracy threshold 1e-06)
Symmetrizer_Uirr initialized for ikirr=10, kpt=1

/home/caua/miniconda3/envs/wb_env3/lib/python3.11/site-packages/wannierberri/symmetry/sawf_kirr.py:163: UserWarning: Warning: max error in block 51 [52:53] is 1.2163475448501176e-06, exceeding threshold 1e-06, and this is not among the  upper bands(56:60) bands, this may indicate inaccuracy in the input data
  warnings.warn(f"Warning: max error in block {i} [{start}:{end}] is {max_error_in_blocks[i]}, exceeding threshold {accuracy_threshold}, and this is not among the  upper"
/home/caua/miniconda3/envs/wb_env3/lib/python3.11/site-packages/wannierberri/symmetry/sawf_kirr.py:163: UserWarning: Warning: max error in block 52 [53:54] is 1.2010930154254992e-06, exceeding threshold 1e-06, and this is not among the  upper bands(56:60) bands, this may indicate inaccuracy in the input data
  warnings.warn(f"Warning: max error in block {i} [{start}:{end}] is {max_error_in_blocks[i]}, exceeding threshold {accuracy_threshold}, and this is not among the  upper"
/home/caua/miniconda3/envs/wb_env3/lib

Symmetrizer_Uirr initialized for ikirr=13, kpt=13, [0.  0.5 0.5] with 8 symmetries, max error in included blocks: 6.828048127635941e-06 ; excluded bands are [57 58 59] out of 60 total bands (accuracy threshold 1e-06)
Symmetrizer_Uirr initialized for ikirr=14, kpt=14, [0.16666667 0.16666667 0.        ] with 4 symmetries, max error in included blocks: 3.0207386053015565e-05 ; excluded bands are [56 57 58 59] out of 60 total bands (accuracy threshold 1e-06)
Symmetrizer_Uirr initialized for ikirr=16, kpt=16, [0.16666667 0.16666667 0.5       ] with 4 symmetries, max error in included blocks: 2.0646624649823727e-05 ; excluded bands are [56 57 58 59] out of 60 total bands (accuracy threshold 1e-06)
Symmetrizer_Uirr initialized for ikirr=23, kpt=23, [0.33333333 0.33333333 0.5       ] with 12 symmetries, max error in included blocks: 0.00016377343559238297 ; excluded bands are [55 56 57 58 59] out of 60 total bands (accuracy threshold 1e-06)
#####################################################

/home/caua/miniconda3/envs/wb_env3/lib/python3.11/site-packages/wannierberri/symmetry/sawf_kirr.py:163: UserWarning: Warning: max error in block 36 [54:55] is 0.00016377343559238297, exceeding threshold 1e-06, and this is not among the  upper bands(56:60) bands, this may indicate inaccuracy in the input data
  warnings.warn(f"Warning: max error in block {i} [{start}:{end}] is {max_error_in_blocks[i]}, exceeding threshold {accuracy_threshold}, and this is not among the  upper"


####################################################################################################
Iteration 0 (from wannierizer)
----------------------------------------------------------------------------------------------------
wannier centers and spreads
----------------------------------------------------------------------------------------------------
  0.000000000000    2.273270496913    1.858788156660   |     2.690134379301
  0.000000000000    2.298712701597    1.539077654835   |     2.766390106016
 -0.000000000000    2.247828292228    1.539077654835   |     2.766390106016
  1.968710000000    1.136635248456    4.794536843340   |     2.690134379301
  1.968710000000    1.111193043772    5.114247345165   |     2.766390106016
  1.968710000000    1.162077453141    5.114247345165   |     2.766390106016
  0.000000000000    0.000000000000    0.000000000000   |     2.346573044634
  0.000000000000    0.000000000000    0.000000000000   |     2.112102174628
  0.000000000000    0.00000000

Now, lets interpolate and get the HR file to use in Wannier Tools!

In [5]:
from wannierberri.system.interpolate import SystemInterpolatorSOC
from wannierberri.system import System_R
interpolator = SystemInterpolatorSOC(system_list[0], system_list[1])
new_system = interpolator.interpolate(alpha/2)
System_R.to_hr_file(new_system,seedname=f'interpolation-{N}')
